In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [452]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(x.year, x.month, 19, tzinfo=timezone)    
    utc_to = datetime(x.year, x.month, x.day+1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
#     rates_frame['sma'] = rates_frame['close'].rolling(window=300).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=30).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [453]:
def get_rsi(close, lookback):
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
    return rsi_df

In [514]:
symbol = "GBPUSD"
a= get_values(symbol)
a['rsi'] = get_rsi(a['close'], 14)

# a['smaL']= a['rsi'].rolling(window=2).mean()
# a['slope'] = go(a)
a = a.dropna()
# a = a[300:]
a

,open,close,rsi
time,,,
2021-08-19 00:01:00,1.37510,1.37506,0.000000
2021-08-19 00:02:00,1.37487,1.37514,39.857651
2021-08-19 00:03:00,1.37533,1.37531,68.546886
2021-08-19 00:04:00,1.37531,1.37523,55.201894
2021-08-19 00:05:00,1.37524,1.37505,37.508048
...,...,...,...
2021-08-20 23:50:00,1.36236,1.36226,30.025224
2021-08-20 23:51:00,1.36227,1.36237,41.244920
2021-08-20 23:52:00,1.36236,1.36232,38.243292


In [515]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

for i in range(5, len(a)):
        if a.iloc[i-3].rsi < a.iloc[i-2].rsi and a.iloc[i-1].rsi < a.iloc[i-2].rsi \
            and a.iloc[i-1].rsi < a.iloc[i].rsi and a.iloc[i].rsi < a.iloc[i-2].rsi \
            and a.iloc[i].close > a.iloc[i].open \
            and (a.iloc[i].rsi - a.iloc[i-1].rsi) >= 2.0 \
             and check == 0:
#             and (a.iloc[i-2].rsi - a.iloc[i-3].rsi) >= 2.0
            
            buy_price = a.iloc[i+1].open
            print("#"*20)
            print(a.iloc[i+1].name)
            print("*"*20)
            check = 1  
            up = 0
            k = 0.0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(f"{pp}---{round(a.iloc[i].rsi, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")
            
            if pp >= 0.0:
                profit.append(pp)
                check = 0
#             elif pp < 0.0:
#                 up = up+1
#                 if up > 1: 
#                     profit.append(pp)
#                     check = 0

#             elif pp < -2.0:
#                 profit.append(pp)
#                 check = 0

            elif pp < 0.0:
                up = up+1
                if up == 1:
                    k = pp
                if up > 1:
                    if pp > k:
                        print("pass")
                    else:
                        profit.append(pp)
                        check = 0
                k = pp
                    #In live trade if loss still goes on to increase then close the trade before hand

####################
2021-08-19 00:22:00
********************
9.0---46.02---1.37506--2021-08-19 00:22:00
####################
2021-08-19 00:28:00
********************
-3.0---49.98---1.37516--2021-08-19 00:28:00
5.0---46.88---1.37508--2021-08-19 00:29:00
####################
2021-08-19 01:01:00
********************
1.0---60.28---1.37533--2021-08-19 01:01:00
####################
2021-08-19 03:09:00
********************
12.0---52.16---1.3748--2021-08-19 03:09:00
####################
2021-08-19 03:27:00
********************
0.0---29.57---1.37378--2021-08-19 03:27:00
####################
2021-08-19 04:00:00
********************
3.0---40.47---1.37337--2021-08-19 04:00:00
####################
2021-08-19 04:33:00
********************
-1.0---63.84---1.37381--2021-08-19 04:33:00
-1.0---63.84---1.37381--2021-08-19 04:34:00
####################
2021-08-19 05:54:00
********************
-1.0---50.37---1.37207--2021-08-19 05:54:00
-6.0---53.18---1.37212--2021-08-19 05:55:00
####################
2021-

In [516]:
sum(profit)

54.0

In [517]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->-317.0
Total negative -->13
Total positive sm -->371.0
Total positive -->37
Length 50
